In [ ]:
import os
import pandas as pd
import re

In [ ]:
folder_path = r"C:\Users\Yang Qing\OneDrive\Desktop\ADS Evaluation\Cleared Transcript by Luke and Chuan"

# Initialize an empty list to store file data
data = []

# Iterate over all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt"):
        file_path = os.path.join(folder_path, file_name)
        
        # Read file content
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()
        
        # Extract the first 16 characters of the file name
        short_name = file_name[:16]

        # Count words
        words = content.split()
        total_words = len(words)

        # Count occurrences based on given criteria
        doctor_count = sum(1 for word in words if re.search(r"doctor", word, re.IGNORECASE))
        patient_count = sum(1 for word in words if re.search(r"patient", word, re.IGNORECASE))
        substitution_count = sum(1 for word in words if "*(S:" in word)
        deletion_count = sum(1 for word in words if "*(D:" in word)
        insertion_count = sum(1 for word in words if "*(I:" in word)
        diarization_error_patients = sum(1 for word in words if 'Patient")*' in word) + sum(1 for word in words if 'Patient)*' in word)
        diarization_error_doctor = sum(1 for word in words if 'Doctor")*' in word) + sum(1 for word in words if 'Doctor)*' in word)
        
        # Append data to the list
        data.append([
            short_name, total_words, doctor_count, patient_count, 
            substitution_count, deletion_count, insertion_count, 
            diarization_error_patients, diarization_error_doctor
        ])

# Create DataFrame
columns = [
    "File Name", "Total Words", "Doctor Count", "Patient Count", 
    "Substitution", "Deletion", "Insertion", 
    "Diarization Error Patients", "Diarization Error Doctor"
]
df = pd.DataFrame(data, columns=columns)

In [ ]:
df.head(39)

,File Name,Total Words,Doctor Count,Patient Count,Substitution,Deletion,Insertion,Diarization Error Patients,Diarization Error Doctor
0,DC0001 - DPt0411,2287,95,89,2,0,1,12,12
1,DC0026 - DPt0326,3392,111,150,3,0,0,3,3
2,DC0026 - DPt0529,1606,70,74,2,1,2,2,1
3,DC0027 - DPt0596,2859,90,106,1,7,2,2,0
4,DC0028 - DPt0506,1219,64,62,0,0,1,0,1
5,DC0033 - DPt0530,2783,114,110,3,1,0,5,7
6,DC0033 - DPt0564,1552,62,100,0,0,0,0,1
7,DC0034 - DPt0329,6606,376,368,1,2,3,6,7
8,DC0042 - DPt0510,3378,107,106,1,0,0,3,2
9,DC0042 - DPt0527,1354,68,69,0,1,0,2,1


In [ ]:
df["WER"] = (df["Substitution"] + df["Deletion"] + df["Insertion"]) / (
    df["Total Words"] - df["Doctor Count"] - df["Patient Count"] - df["Substitution"]
    - df["Deletion"] - df["Insertion"] - df["Diarization Error Patients"] - df["Diarization Error Doctor"]
)
print(df["WER"].mean())

0.002075885240904833


In [ ]:
df["Diarization Error"] = (df["Diarization Error Patients"] + df["Diarization Error Doctor"]) / (
    df["Doctor Count"] + df["Patient Count"] - df["Diarization Error Patients"] - df["Diarization Error Doctor"]
)
print(1-df["Diarization Error"].mean())

0.961902592350579
